## DP1 optical--NIR source statistics

This notebook quantifies source populations in the final reduced DP1
LSST--VISTA catalogue for the Discussion section of the LSST--VISTA
Data Fusion paper.

The analysis uses forced circular-aperture photometry at the common
source positions. A significant measurement is defined by
S/N >= 5. Particular attention is given to sources with
significant measurements in multiple bands in one wavelength regime
while remaining below this threshold in the other.

### 1. Imports and catalogue

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from astropy.table import Table


# Final reduced DP1 catalogue.
DATA_DIR = Path("../data")

FINAL_REDUCED_CATALOGUE = (
    DATA_DIR / "full_reduced_cat_DP1_20260902.fits"
)

OUT_DIR = Path("data")

if not FINAL_REDUCED_CATALOGUE.exists():
    raise FileNotFoundError(
        f"Catalogue not found: {FINAL_REDUCED_CATALOGUE}"
    )

cat = Table.read(FINAL_REDUCED_CATALOGUE)

print("Catalogue:", FINAL_REDUCED_CATALOGUE)
print(f"Rows      : {len(cat):,}")
print(f"Columns   : {len(cat.colnames):,}")

Catalogue: ../data/full_reduced_cat_DP1_20260902.fits
Rows      : 1,380,173
Columns   : 274


### 2. Forced-photometry columns

Verify that the reduced catalogue contains the forced aperture flux,
uncertainty, and measurement flag required in all eleven bands.

In [2]:
COMCAM_BANDS = ["u", "g", "r", "i", "z", "y"]
VIRCAM_BANDS = ["Z", "Y", "J", "H", "K"]

ALL_BANDS = COMCAM_BANDS + VIRCAM_BANDS


# Severe PixelFlags used for the clean forced-photometry selection.
BAD_PIXEL_FLAGS = [
    "offimage",
    "edgeCenterAll",
    "nodataCenterAll",
    "interpolatedCenterAll",
    "saturatedCenterAll",
    "crCenterAll",
    "badCenterAll",
    "clippedCenterAll",
    "rejectedCenterAll",
]


required_columns = []

for band in COMCAM_BANDS:

    prefix = f"ComCam_{band}_f"

    stem = (
        f"{prefix}_"
        "base_CircularApertureFlux_6_0"
    )

    required_columns.extend([
        f"{stem}_flux",
        f"{stem}_fluxErr",
        f"{stem}_flag",
    ])

    required_columns.extend([
        f"{prefix}_base_PixelFlags_flag_{flag}"
        for flag in BAD_PIXEL_FLAGS
    ])


for band in VIRCAM_BANDS:

    prefix = f"VIRCAM_{band}_f"

    stem = (
        f"{prefix}_"
        "base_CircularApertureFlux_6_0"
    )

    required_columns.extend([
        f"{stem}_flux",
        f"{stem}_fluxErr",
        f"{stem}_flag",
    ])

    required_columns.extend([
        f"{prefix}_base_PixelFlags_flag_{flag}"
        for flag in BAD_PIXEL_FLAGS
    ])


missing = [
    name
    for name in required_columns
    if name not in cat.colnames
]


if missing:

    raise RuntimeError(
        "Missing forced-photometry or PixelFlag columns:\n"
        + "\n".join(missing)
    )


print(
    "All required forced-photometry and "
    "PixelFlag columns are present."
)

All required forced-photometry and PixelFlag columns are present.


### 3. Forced-photometry validity and signal-to-noise

A forced measurement is considered valid when its flux and uncertainty are finite, the uncertainty is positive, the circular-aperture measurement flag is not set, and none of the selected severe pixel-quality flags is set. Signal-to-noise is calculated as flux / flux uncertainty, with S/N >= 5 used to identify significant measurements.

The quality selection is applied independently in every band. Measurements are not required to be significant to be valid; a clean low-significance forced measurement can still provide useful flux information or an upper-limit constraint.

In [3]:
SNR_THRESHOLD = 5.0


def as_float(column):
    return np.asarray(
        np.ma.filled(column, np.nan),
        dtype=float,
    )


def as_flag(column):
    return np.asarray(
        np.ma.filled(column, True),
        dtype=bool,
    )


def get_forced_measurement(cat, instrument, band):

    prefix = f"{instrument}_{band}_f"

    stem = (
        f"{prefix}_"
        "base_CircularApertureFlux_6_0"
    )

    flux = as_float(
        cat[f"{stem}_flux"]
    )

    flux_err = as_float(
        cat[f"{stem}_fluxErr"]
    )

    aperture_flag = as_flag(
        cat[f"{stem}_flag"]
    )


    # Combine the selected severe PixelFlags.
    bad_pixel = np.zeros(
        len(cat),
        dtype=bool,
    )

    for flag_name in BAD_PIXEL_FLAGS:

        column = (
            f"{prefix}_"
            f"base_PixelFlags_flag_{flag_name}"
        )

        bad_pixel |= as_flag(
            cat[column]
        )


    valid = (
        np.isfinite(flux)
        & np.isfinite(flux_err)
        & (flux_err > 0)
        & (~aperture_flag)
        & (~bad_pixel)
    )


    snr = np.full(
        len(cat),
        np.nan,
    )

    snr[valid] = (
        flux[valid]
        / flux_err[valid]
    )


    significant = (
        valid
        & (snr >= SNR_THRESHOLD)
    )


    return {
        "flux": flux,
        "flux_err": flux_err,
        "aperture_flag": aperture_flag,
        "bad_pixel": bad_pixel,
        "valid": valid,
        "snr": snr,
        "significant": significant,
    }


forced = {}


for band in COMCAM_BANDS:

    forced[f"ComCam_{band}"] = (
        get_forced_measurement(
            cat,
            "ComCam",
            band,
        )
    )


for band in VIRCAM_BANDS:

    forced[f"VIRCAM_{band}"] = (
        get_forced_measurement(
            cat,
            "VIRCAM",
            band,
        )
    )


print(
    "Forced photometry constructed with "
    "aperture and PixelFlag quality selection."
)

Forced photometry constructed with aperture and PixelFlag quality selection.


### 4. Per-band forced-photometry statistics

Count the number of valid forced measurements and measurements with
S/N >= 5 in each optical and NIR band. A valid measurement satisfies
the forced-photometry quality criteria defined above, including the
aperture measurement flag and selected severe pixel-quality flags.

In [4]:
print(f"Total catalogue rows: {len(cat):,}\n")

print("ComCam")
for band in COMCAM_BANDS:
    m = forced[f"ComCam_{band}"]

    print(
        f"  {band:>2s}: "
        f"valid = {m['valid'].sum():>8,d}   "
        f"S/N >= 5 = {m['significant'].sum():>8,d}"
    )

print("\nVIRCAM")
for band in VIRCAM_BANDS:
    m = forced[f"VIRCAM_{band}"]

    print(
        f"  {band:>2s}: "
        f"valid = {m['valid'].sum():>8,d}   "
        f"S/N >= 5 = {m['significant'].sum():>8,d}"
    )

Total catalogue rows: 1,380,173

ComCam
   u: valid =  722,246   S/N >= 5 =   60,796
   g: valid =  611,524   S/N >= 5 =  287,672
   r: valid =  591,187   S/N >= 5 =  287,517
   i: valid =  619,012   S/N >= 5 =  261,688
   z: valid =  639,428   S/N >= 5 =  186,641
   y: valid =  701,121   S/N >= 5 =   73,785

VIRCAM
   Z: valid =  283,360   S/N >= 5 =  172,158
   Y: valid =  978,586   S/N >= 5 =  730,007
   J: valid = 1,036,437   S/N >= 5 =  672,680
   H: valid =  963,692   S/N >= 5 =  581,460
   K: valid = 1,020,741   S/N >= 5 =  511,551


### 5. Primary-source multi-band statistics

Restrict the analysis to sources with `detect_isPrimary = 1` and count
how many have significant clean forced photometry in multiple ComCam and
VIRCAM bands. A significant measurement satisfies the forced-photometry
quality criteria defined above and has S/N >= 5.

In [5]:
PRIMARY_COLUMN = "VIRCAM_K_m_detect_isPrimary"

primary = np.asarray(
    np.ma.filled(cat[PRIMARY_COLUMN], False),
    dtype=bool,
)


optical_valid = np.column_stack([
    forced[f"ComCam_{band}"]["valid"]
    for band in COMCAM_BANDS
])

optical_significant = np.column_stack([
    forced[f"ComCam_{band}"]["significant"]
    for band in COMCAM_BANDS
])

nir_valid = np.column_stack([
    forced[f"VIRCAM_{band}"]["valid"]
    for band in VIRCAM_BANDS
])

nir_significant = np.column_stack([
    forced[f"VIRCAM_{band}"]["significant"]
    for band in VIRCAM_BANDS
])


n_optical_valid = optical_valid.sum(axis=1)
n_optical_5sig = optical_significant.sum(axis=1)

n_nir_valid = nir_valid.sum(axis=1)
n_nir_5sig = nir_significant.sum(axis=1)


at_least_two_optical = (
    n_optical_5sig >= 2
)

at_least_two_nir = (
    n_nir_5sig >= 2
)

both_regimes = (
    primary
    & at_least_two_optical
    & at_least_two_nir
)


print("Catalogue population")
print("--------------------")
print(f"All catalogue rows : {len(cat):,}")
print(f"Primary sources    : {primary.sum():,}")

print("\nSignificant forced photometry")
print("-----------------------------")
print(
    ">=2 ComCam bands at 5 sigma :",
    f"{np.sum(primary & at_least_two_optical):,}",
)
print(
    ">=2 VIRCAM bands at 5 sigma :",
    f"{np.sum(primary & at_least_two_nir):,}",
)
print(
    ">=2 bands in both regimes   :",
    f"{np.sum(both_regimes):,}",
)

Catalogue population
--------------------
All catalogue rows : 1,380,173
Primary sources    : 614,223

Significant forced photometry
-----------------------------
>=2 ComCam bands at 5 sigma : 151,318
>=2 VIRCAM bands at 5 sigma : 318,488
>=2 bands in both regimes   : 116,036


### 6. Extreme optical-NIR populations

To distinguish low-significance measurements from missing coverage, the
NIR-selected sample is required to have valid forced measurements in the
ComCam g, r, i, and z bands, while the optically selected sample is
required to have valid forced measurements in the VIRCAM Y, J, H, and
Ks bands. The more limited ComCam u and y and VIRCAM Z coverage is not
required.

The extreme populations are defined as primary sources with S/N >= 5
in at least two bands in one wavelength regime and no valid forced
measurement with |S/N| >= 5 in the other regime. The full set of bands
is considered when testing the latter condition.

In [6]:
COMCAM_CORE_BANDS = ["g", "r", "i", "z"]
VIRCAM_CORE_BANDS = ["Y", "J", "H", "K"]


# Require clean forced-photometry coverage in the core bands.
all_optical_core_valid = np.column_stack([
    forced[f"ComCam_{band}"]["valid"]
    for band in COMCAM_CORE_BANDS
]).all(axis=1)

all_nir_core_valid = np.column_stack([
    forced[f"VIRCAM_{band}"]["valid"]
    for band in VIRCAM_CORE_BANDS
]).all(axis=1)


# Does any valid measurement in the opposite regime have |S/N| >= 5?
optical_abs_5sig = np.column_stack([
    forced[f"ComCam_{band}"]["valid"]
    & (
        np.abs(forced[f"ComCam_{band}"]["snr"])
        >= SNR_THRESHOLD
    )
    for band in COMCAM_BANDS
]).any(axis=1)

nir_abs_5sig = np.column_stack([
    forced[f"VIRCAM_{band}"]["valid"]
    & (
        np.abs(forced[f"VIRCAM_{band}"]["snr"])
        >= SNR_THRESHOLD
    )
    for band in VIRCAM_BANDS
]).any(axis=1)


nir_extreme = (
    primary
    & all_optical_core_valid
    & (n_nir_5sig >= 2)
    & (~optical_abs_5sig)
)

optical_extreme = (
    primary
    & all_nir_core_valid
    & (n_optical_5sig >= 2)
    & (~nir_abs_5sig)
)


print("Core-band coverage")
print("------------------")
print(
    "Primary sources with valid griz  :",
    f"{np.sum(primary & all_optical_core_valid):,}",
)
print(
    "Primary sources with valid YJHKs :",
    f"{np.sum(primary & all_nir_core_valid):,}",
)

print("\nExtreme populations")
print("-------------------")
print(
    "NIR significant / all ComCam |S/N| < 5 :",
    f"{np.sum(nir_extreme):,}",
)
print(
    "ComCam significant / all VIRCAM |S/N| < 5:",
    f"{np.sum(optical_extreme):,}",
)

Core-band coverage
------------------
Primary sources with valid griz  : 199,294
Primary sources with valid YJHKs : 410,731

Extreme populations
-------------------
NIR significant / all ComCam |S/N| < 5 : 24,222
ComCam significant / all VIRCAM |S/N| < 5: 3,458


### 7. Frequency of extreme optical-NIR sources

Calculate the size of the parent populations satisfying the appropriate
coverage requirement and having significant forced measurements in at
least two bands. The extreme fraction is then measured as the fraction
of each parent population with no valid forced measurement having
|S/N| >= 5 in the opposite wavelength regime.

In [7]:
# NIR-selected parent population:
# valid ComCam griz coverage and >=2 significant VIRCAM bands.
nir_parent = (
    primary
    & all_optical_core_valid
    & (n_nir_5sig >= 2)
)

# Optical-selected parent population:
# valid VIRCAM YJHKs coverage and >=2 significant ComCam bands.
optical_parent = (
    primary
    & all_nir_core_valid
    & (n_optical_5sig >= 2)
)

n_nir_parent = np.sum(nir_parent)
n_optical_parent = np.sum(optical_parent)

n_nir_extreme = np.sum(nir_extreme)
n_optical_extreme = np.sum(optical_extreme)

print("NIR-selected population")
print("-----------------------")
print(f"Parent sample               : {n_nir_parent:,}")
print(f"All ComCam |S/N| < 5        : {n_nir_extreme:,}")
print(
    f"Extreme fraction            : "
    f"{100 * n_nir_extreme / n_nir_parent:.2f}%"
)

print("\nComCam-selected population")
print("--------------------------")
print(f"Parent sample               : {n_optical_parent:,}")
print(f"All VIRCAM |S/N| < 5        : {n_optical_extreme:,}")
print(
    f"Extreme fraction            : "
    f"{100 * n_optical_extreme / n_optical_parent:.2f}%"
)

NIR-selected population
-----------------------
Parent sample               : 104,886
All ComCam |S/N| < 5        : 24,222
Extreme fraction            : 23.09%

ComCam-selected population
--------------------------
Parent sample               : 110,959
All VIRCAM |S/N| < 5        : 3,458
Extreme fraction            : 3.12%


### 8. Strong-contrast candidates for visual inspection

Construct cleaner subsets of the extreme populations for visual inspection.
These selections are used only to identify illustrative examples and do not
modify the statistical populations or fractions defined above.

Sources must have |S/N| < 3 in every valid forced measurement in the
opposite wavelength regime. For visual inspection, the core bands are also
required to be free of edge and missing-data flags, and sources are ranked
after requiring at least 6 arcsec separation from the nearest other primary
catalogue source.

In [8]:
# ============================================================
# Strong-contrast candidates for visual inspection
# ============================================================

from astropy.coordinates import SkyCoord
import astropy.units as u


STRONG_OPPOSITE_SNR = 3.0
ISOLATION_ARCSEC = 6.0
INSPECTION_TOP_N = 20


# ------------------------------------------------------------
# Coordinates
# ------------------------------------------------------------

ra_rad = np.asarray(
    np.ma.filled(
        cat["VIRCAM_K_m_coord_ra"],
        np.nan,
    ),
    dtype=float,
)

dec_rad = np.asarray(
    np.ma.filled(
        cat["VIRCAM_K_m_coord_dec"],
        np.nan,
    ),
    dtype=float,
)

ra_deg = np.degrees(ra_rad)
dec_deg = np.degrees(dec_rad)

finite_coord = (
    np.isfinite(ra_deg)
    & np.isfinite(dec_deg)
)


# ------------------------------------------------------------
# S/N arrays
# ------------------------------------------------------------

optical_snr = np.column_stack(
    [
        forced[f"ComCam_{band}"]["snr"]
        for band in COMCAM_BANDS
    ]
)

nir_snr = np.column_stack(
    [
        forced[f"VIRCAM_{band}"]["snr"]
        for band in VIRCAM_BANDS
    ]
)


n_optical_valid = optical_valid.sum(axis=1)
n_nir_valid = nir_valid.sum(axis=1)

n_valid_all = (
    n_optical_valid
    + n_nir_valid
)


# ------------------------------------------------------------
# Maximum absolute S/N among valid measurements
# ------------------------------------------------------------

def max_abs_valid_snr(
    snr_array,
    valid_array,
):

    result = np.full(
        len(cat),
        np.nan,
    )

    has_valid = valid_array.any(
        axis=1
    )

    values = np.where(
        valid_array[has_valid],
        np.abs(
            snr_array[has_valid]
        ),
        np.nan,
    )

    result[has_valid] = np.nanmax(
        values,
        axis=1,
    )

    return result


max_abs_optical_snr = max_abs_valid_snr(
    optical_snr,
    optical_valid,
)

max_abs_nir_snr = max_abs_valid_snr(
    nir_snr,
    nir_valid,
)


# ------------------------------------------------------------
# Strong optical--NIR contrast populations
#
# These are subsets of the extreme populations above:
# >=2 significant bands on one side,
# all valid measurements on the other side below |S/N| = 3.
# ------------------------------------------------------------

nir_strong = (
    primary
    & all_optical_core_valid
    & (n_nir_5sig >= 2)
    & (
        max_abs_optical_snr
        < STRONG_OPPOSITE_SNR
    )
)

optical_strong = (
    primary
    & all_nir_core_valid
    & (n_optical_5sig >= 2)
    & (
        max_abs_nir_snr
        < STRONG_OPPOSITE_SNR
    )
)


# ------------------------------------------------------------
# Image-quality screening for visual examples
#
# Apply only to the well-covered core bands.
# ------------------------------------------------------------

CORE_VISUAL_BANDS = {
    "ComCam": [
        "g",
        "r",
        "i",
        "z",
    ],
    "VIRCAM": [
        "Y",
        "J",
        "H",
        "K",
    ],
}


VISUAL_EDGE_FLAGS = [
    "offimage",
    "edge",
    "edgeCenter",
    "edgeCenterAll",
    "sensor_edge",
    "sensor_edgeCenter",
    "sensor_edgeCenterAll",
    "nodata",
    "nodataCenter",
    "nodataCenterAll",
]


visual_pixel_clean = np.ones(
    len(cat),
    dtype=bool,
)


for instrument, bands in (
    CORE_VISUAL_BANDS.items()
):

    for band in bands:

        prefix = (
            f"{instrument}_{band}_f"
        )

        for flag_name in (
            VISUAL_EDGE_FLAGS
        ):

            column = (
                f"{prefix}_"
                f"base_PixelFlags_flag_"
                f"{flag_name}"
            )

            if column in cat.colnames:

                visual_pixel_clean &= (
                    ~as_flag(
                        cat[column]
                    )
                )


# ------------------------------------------------------------
# Separation from nearest OTHER primary source
# ------------------------------------------------------------

primary_index = np.flatnonzero(
    primary
    & finite_coord
)

primary_coord = SkyCoord(
    ra=ra_deg[
        primary_index
    ] * u.deg,
    dec=dec_deg[
        primary_index
    ] * u.deg,
)


candidate_index = np.flatnonzero(
    (
        nir_strong
        | optical_strong
    )
    & finite_coord
)

candidate_coord = SkyCoord(
    ra=ra_deg[
        candidate_index
    ] * u.deg,
    dec=dec_deg[
        candidate_index
    ] * u.deg,
)


# Candidates are themselves primary sources, so
# nthneighbor=2 gives the nearest OTHER primary source.
_, nearest_sep, _ = (
    candidate_coord
    .match_to_catalog_sky(
        primary_coord,
        nthneighbor=2,
    )
)


nearest_primary_sep = np.full(
    len(cat),
    np.nan,
)

nearest_primary_sep[
    candidate_index
] = nearest_sep.arcsec


visual_quality = (
    visual_pixel_clean
    & np.isfinite(
        nearest_primary_sep
    )
    & (
        nearest_primary_sep
        > ISOLATION_ARCSEC
    )
)


nir_visual = (
    nir_strong
    & visual_quality
)

optical_visual = (
    optical_strong
    & visual_quality
)


# ------------------------------------------------------------
# Median S/N on the significantly measured side
# ------------------------------------------------------------

def median_significant_snr(
    snr_array,
    significant_array,
):

    result = np.full(
        len(cat),
        np.nan,
    )

    has_significant = (
        significant_array.any(
            axis=1
        )
    )

    values = np.where(
        significant_array[
            has_significant
        ],
        snr_array[
            has_significant
        ],
        np.nan,
    )

    result[
        has_significant
    ] = np.nanmedian(
        values,
        axis=1,
    )

    return result


median_optical_snr = (
    median_significant_snr(
        optical_snr,
        optical_significant,
    )
)

median_nir_snr = (
    median_significant_snr(
        nir_snr,
        nir_significant,
    )
)


# ------------------------------------------------------------
# Helper for candidate tables
# ------------------------------------------------------------

def build_candidate_table(
    mask,
    sample,
):

    idx = np.flatnonzero(
        mask
    )


    if sample == "NIR_significant":

        n_sig = (
            n_nir_5sig[idx]
        )

        median_sig = (
            median_nir_snr[idx]
        )

        max_opposite = (
            max_abs_optical_snr[
                idx
            ]
        )

    else:

        n_sig = (
            n_optical_5sig[idx]
        )

        median_sig = (
            median_optical_snr[
                idx
            ]
        )

        max_opposite = (
            max_abs_nir_snr[
                idx
            ]
        )


    data = {
        "catalog_row":
            idx,

        "sample":
            sample,

        "ra_deg":
            ra_deg[idx],

        "dec_deg":
            dec_deg[idx],

        "n_valid_bands":
            n_valid_all[idx],

        "n_significant_bands":
            n_sig,

        "median_significant_snr":
            median_sig,

        "max_abs_opposite_snr":
            max_opposite,

        "nearest_primary_sep_arcsec":
            nearest_primary_sep[idx],
    }


    # Store all 11 forced-photometry S/N values
    for band in COMCAM_BANDS:

        data[
            f"snr_ComCam_{band}"
        ] = (
            forced[
                f"ComCam_{band}"
            ]["snr"][idx]
        )


    for band in VIRCAM_BANDS:

        data[
            f"snr_VIRCAM_{band}"
        ] = (
            forced[
                f"VIRCAM_{band}"
            ]["snr"][idx]
        )


    return pd.DataFrame(
        data
    )


# ------------------------------------------------------------
# Build and rank the visual-inspection populations
# ------------------------------------------------------------

nir_candidates = (
    build_candidate_table(
        nir_visual,
        "NIR_significant",
    )
)

optical_candidates = (
    build_candidate_table(
        optical_visual,
        "ComCam_significant",
    )
)


strong_candidates = pd.concat(
    [
        optical_candidates,
        nir_candidates,
    ],
    ignore_index=True,
)


strong_candidates = (
    strong_candidates
    .sort_values(
        by=[
            "sample",
            "n_significant_bands",
            "nearest_primary_sep_arcsec",
            "median_significant_snr",
            "max_abs_opposite_snr",
        ],
        ascending=[
            True,
            False,
            False,
            False,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)


inspection_candidates = (
    strong_candidates
    .groupby(
        "sample",
        group_keys=False,
    )
    .head(
        INSPECTION_TOP_N
    )
    .reset_index(
        drop=True
    )
)


inspection_candidates[
    "inspection_id"
] = (
    np.arange(
        len(
            inspection_candidates
        )
    )
    + 1
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


ALL_STRONG_FILE = (
    OUT_DIR
    / "DP1_strong_contrast_sources_forced.csv"
)

RSP_INSPECTION_FILE = (
    OUT_DIR
    / "DP1_strong_contrast_inspection_forced.csv"
)


strong_candidates.to_csv(
    ALL_STRONG_FILE,
    index=False,
)

inspection_candidates.to_csv(
    RSP_INSPECTION_FILE,
    index=False,
)


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print(
    "Strong-contrast populations"
)

print(
    "---------------------------"
)

print(
    "NIR significant       :",
    f"{nir_strong.sum():,}",
)

print(
    "ComCam significant    :",
    f"{optical_strong.sum():,}",
)


print(
    "\nAfter visual-quality "
    "and isolation cuts"
)

print(
    "---------------------------------------"
)

print(
    "NIR significant       :",
    f"{nir_visual.sum():,}",
)

print(
    "ComCam significant    :",
    f"{optical_visual.sum():,}",
)


print(
    "\nRSP inspection sample"
)

print(
    "---------------------"
)

print(
    inspection_candidates[
        "sample"
    ].value_counts()
)


print("\nSaved:")
print(ALL_STRONG_FILE)
print(RSP_INSPECTION_FILE)


display(
    inspection_candidates[
        [
            "inspection_id",
            "catalog_row",
            "sample",
            "ra_deg",
            "dec_deg",
            "n_valid_bands",
            "n_significant_bands",
            "median_significant_snr",
            "max_abs_opposite_snr",
            "nearest_primary_sep_arcsec",
        ]
    ]
)

Strong-contrast populations
---------------------------
NIR significant       : 8,439
ComCam significant    : 1,054

After visual-quality and isolation cuts
---------------------------------------
NIR significant       : 445
ComCam significant    : 4

RSP inspection sample
---------------------
sample
NIR_significant       20
ComCam_significant     4
Name: count, dtype: int64

Saved:
data/DP1_strong_contrast_sources_forced.csv
data/DP1_strong_contrast_inspection_forced.csv


,inspection_id,catalog_row,sample,ra_deg,dec_deg,n_valid_bands,n_significant_bands,median_significant_snr,max_abs_opposite_snr,nearest_primary_sep_arcsec
0,1,163503,ComCam_significant,53.621950,-27.659692,9,3,6.056666,2.207638,6.092301
1,2,573825,ComCam_significant,53.604866,-27.840971,10,2,5.694068,2.001595,8.568890
2,3,607371,ComCam_significant,53.720010,-28.046087,10,2,6.657598,2.090794,6.160234
3,4,883070,ComCam_significant,53.467985,-28.427412,10,2,5.985270,1.410415,6.002824
4,5,70179,NIR_significant,52.477046,-27.830815,9,5,20.118377,1.598241,9.754040
5,6,666836,NIR_significant,52.653276,-27.633525,11,5,16.247038,2.415569,9.236922
6,7,252997,NIR_significant,52.468875,-28.037428,9,5,19.939622,2.393062,8.197731
7,8,57122,NIR_significant,52.455010,-27.986315,9,5,44.895305,2.718032,8.128705
8,9,453852,NIR_significant,52.558058,-27.745834,9,5,15.140711,2.634865,7.941971
9,10,197408,NIR_significant,52.862614,-27.546313,11,5,10.319294,2.643548,7.802467


### 9. Optical examples for the paper

The general isolation criterion above is useful for producing a clean
inspection sample, but it can reject otherwise suitable optical examples
because of faint nearby catalogue sources. For the final optical examples,
we therefore use a more targeted veto against bright neighbours.

The source must have at least two significant ComCam measurements,
maximum VIRCAM |S/N| < 3, median significant optical S/N >= 7, and
maximum optical S/N >= 10. Sources with another bright primary source
within 10 arcsec are excluded.

In [9]:
# ============================================================
# Optical paper examples:
# veto bright neighbours rather than all nearby sources
# ============================================================

OPT_MAX_NIR_ABS_SNR = 3.0

OPT_MIN_SIG_BANDS = 2
OPT_MIN_MEDIAN_SIG_SNR = 7.0
OPT_MIN_MAX_SNR = 10.0

BRIGHT_NEIGHBOUR_SNR = 30.0
BRIGHT_NEIGHBOUR_RADIUS = 10.0  # arcsec


# ------------------------------------------------------------
# Maximum positive ComCam S/N
# ------------------------------------------------------------

max_optical_snr = np.full(
    len(cat),
    np.nan,
)

has_optical_valid = (
    optical_valid.any(
        axis=1
    )
)

max_optical_snr[
    has_optical_valid
] = np.nanmax(
    np.where(
        optical_valid[
            has_optical_valid
        ],
        optical_snr[
            has_optical_valid
        ],
        np.nan,
    ),
    axis=1,
)


# ------------------------------------------------------------
# Bright-source metric
#
# A primary source is "bright" if it reaches S/N >= 30
# in at least one of the core grizYJHK bands.
# ------------------------------------------------------------

CORE_BRIGHT_KEYS = (
    [
        f"ComCam_{band}"
        for band in [
            "g",
            "r",
            "i",
            "z",
        ]
    ]
    +
    [
        f"VIRCAM_{band}"
        for band in [
            "Y",
            "J",
            "H",
            "K",
        ]
    ]
)


core_bright_snr = np.column_stack(
    [
        forced[key]["snr"]
        for key in CORE_BRIGHT_KEYS
    ]
)

core_bright_valid = np.column_stack(
    [
        forced[key]["valid"]
        for key in CORE_BRIGHT_KEYS
    ]
)


max_core_snr = np.full(
    len(cat),
    np.nan,
)

has_core_valid = (
    core_bright_valid.any(
        axis=1
    )
)

max_core_snr[
    has_core_valid
] = np.nanmax(
    np.where(
        core_bright_valid[
            has_core_valid
        ],
        core_bright_snr[
            has_core_valid
        ],
        np.nan,
    ),
    axis=1,
)


bright_primary = (
    primary
    & finite_coord
    & np.isfinite(
        max_core_snr
    )
    & (
        max_core_snr
        >= BRIGHT_NEIGHBOUR_SNR
    )
)


# ------------------------------------------------------------
# Optical candidate pool before neighbour veto
# ------------------------------------------------------------

optical_base = (
    primary
    & finite_coord

    # clean core-band image environment
    & visual_pixel_clean

    # reliable YJHKs forced-photometry coverage
    & all_nir_core_valid

    # >=2 significant ComCam bands
    & (
        n_optical_5sig
        >= OPT_MIN_SIG_BANDS
    )

    # strong optical measurements
    & (
        median_optical_snr
        >= OPT_MIN_MEDIAN_SIG_SNR
    )
    & (
        max_optical_snr
        >= OPT_MIN_MAX_SNR
    )

    # no significant NIR counterpart
    & (
        max_abs_nir_snr
        < OPT_MAX_NIR_ABS_SNR
    )
)


# ------------------------------------------------------------
# Distance to nearest OTHER bright primary source
# ------------------------------------------------------------

bright_index = np.flatnonzero(
    bright_primary
)

bright_coord = SkyCoord(
    ra=ra_deg[
        bright_index
    ] * u.deg,
    dec=dec_deg[
        bright_index
    ] * u.deg,
)


optical_base_index = np.flatnonzero(
    optical_base
)

optical_base_coord = SkyCoord(
    ra=ra_deg[
        optical_base_index
    ] * u.deg,
    dec=dec_deg[
        optical_base_index
    ] * u.deg,
)


# First and second nearest bright sources
_, sep1, _ = (
    optical_base_coord
    .match_to_catalog_sky(
        bright_coord,
        nthneighbor=1,
    )
)

_, sep2, _ = (
    optical_base_coord
    .match_to_catalog_sky(
        bright_coord,
        nthneighbor=2,
    )
)


# If a candidate itself belongs to the bright-source
# catalogue, the first match is itself.
self_match = (
    sep1.arcsec
    < 0.3
)


nearest_bright_sep_candidate = np.where(
    self_match,
    sep2.arcsec,
    sep1.arcsec,
)


nearest_bright_sep = np.full(
    len(cat),
    np.nan,
)

nearest_bright_sep[
    optical_base_index
] = nearest_bright_sep_candidate


no_bright_neighbour = (
    nearest_bright_sep
    > BRIGHT_NEIGHBOUR_RADIUS
)


optical_paper = (
    optical_base
    & no_bright_neighbour
)


# ------------------------------------------------------------
# Build final optical-example table
# ------------------------------------------------------------

optical_paper_table = (
    build_candidate_table(
        optical_paper,
        "ComCam_significant",
    )
)


optical_paper_table[
    "nearest_bright_sep_arcsec"
] = (
    nearest_bright_sep[
        optical_paper
    ]
)


optical_paper_table = (
    optical_paper_table
    .sort_values(
        by=[
            "nearest_bright_sep_arcsec",
            "n_significant_bands",
            "median_significant_snr",
            "max_abs_opposite_snr",
        ],
        ascending=[
            False,
            False,
            False,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)


optical_paper_table[
    "inspection_id"
] = (
    np.arange(
        len(
            optical_paper_table
        )
    )
    + 1
)


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print(
    "OPTICAL PAPER-EXAMPLE SEARCH"
)

print(
    "----------------------------"
)

print(
    f"VIRCAM max |S/N|          : "
    f"< {OPT_MAX_NIR_ABS_SNR:.1f}"
)

print(
    f"Significant ComCam bands  : "
    f">= {OPT_MIN_SIG_BANDS}"
)

print(
    f"Median optical S/N        : "
    f">= {OPT_MIN_MEDIAN_SIG_SNR:.1f}"
)

print(
    f"Maximum optical S/N       : "
    f">= {OPT_MIN_MAX_SNR:.1f}"
)

print(
    f"Bright-source threshold   : "
    f"S/N >= {BRIGHT_NEIGHBOUR_SNR:.0f}"
)

print(
    f"Bright-neighbour veto     : "
    f"< {BRIGHT_NEIGHBOUR_RADIUS:.1f} arcsec"
)


print(
    "\nCandidates before "
    "bright-neighbour veto :",
    f"{optical_base.sum():,}",
)

print(
    "Candidates after "
    "bright-neighbour veto  :",
    f"{optical_paper.sum():,}",
)


display(
    optical_paper_table[
        [
            "inspection_id",
            "catalog_row",
            "ra_deg",
            "dec_deg",
            "n_valid_bands",
            "n_significant_bands",
            "median_significant_snr",
            "max_abs_opposite_snr",
            "nearest_bright_sep_arcsec",
        ]
    ]
)

OPTICAL PAPER-EXAMPLE SEARCH
----------------------------
VIRCAM max |S/N|          : < 3.0
Significant ComCam bands  : >= 2
Median optical S/N        : >= 7.0
Maximum optical S/N       : >= 10.0
Bright-source threshold   : S/N >= 30
Bright-neighbour veto     : < 10.0 arcsec

Candidates before bright-neighbour veto : 32
Candidates after bright-neighbour veto  : 2


,inspection_id,catalog_row,ra_deg,dec_deg,n_valid_bands,n_significant_bands,median_significant_snr,max_abs_opposite_snr,nearest_bright_sep_arcsec
0,1,15805,53.073812,-27.732826,10,2,14.991056,1.279470,11.515727
1,2,441278,53.593073,-27.926565,8,3,8.452413,1.713349,10.561262


### 10. Final paper examples

Save the small set of sources retained for final image inspection.
The optical examples are the two objects surviving the bright-neighbour
selection above. Two clean NIR strong-contrast sources are retained,
including the source used in the final paper figure.

In [10]:
# ============================================================
# Final paper examples
# ============================================================

FINAL_OPTICAL_ROWS = [
    15805,
    441278,
]

FINAL_NIR_ROWS = [
    666836,
    197408,
]


# ------------------------------------------------------------
# Check that the fixed examples still satisfy
# the intended selections
# ------------------------------------------------------------

for row_index in FINAL_OPTICAL_ROWS:

    if not optical_paper[
        row_index
    ]:

        raise RuntimeError(
            f"Optical final example "
            f"{row_index} no longer satisfies "
            f"the optical paper selection."
        )


for row_index in FINAL_NIR_ROWS:

    if not nir_visual[
        row_index
    ]:

        raise RuntimeError(
            f"NIR final example "
            f"{row_index} no longer satisfies "
            f"the strong NIR visual selection."
        )


# ------------------------------------------------------------
# Build masks
# ------------------------------------------------------------

final_optical_mask = np.zeros(
    len(cat),
    dtype=bool,
)

final_optical_mask[
    FINAL_OPTICAL_ROWS
] = True


final_nir_mask = np.zeros(
    len(cat),
    dtype=bool,
)

final_nir_mask[
    FINAL_NIR_ROWS
] = True


# ------------------------------------------------------------
# Build tables
# ------------------------------------------------------------

final_optical = (
    build_candidate_table(
        final_optical_mask,
        "ComCam_significant",
    )
)

final_nir = (
    build_candidate_table(
        final_nir_mask,
        "NIR_significant",
    )
)


example_candidates = pd.concat(
    [
        final_optical,
        final_nir,
    ],
    ignore_index=True,
)


# ------------------------------------------------------------
# Stable candidate IDs used by the image notebook
#
# 1,2 = optical examples
# 3,4 = NIR examples
# ------------------------------------------------------------

example_candidates[
    "candidate_id"
] = np.arange(
    1,
    len(
        example_candidates
    )
    + 1,
)


# Keep for compatibility with the existing
# inspection workflow.
example_candidates[
    "paper_candidate_id"
] = (
    example_candidates[
        "candidate_id"
    ]
)


# ------------------------------------------------------------
# Column order
# ------------------------------------------------------------

first_columns = [
    "candidate_id",
    "paper_candidate_id",
    "catalog_row",
    "sample",
    "ra_deg",
    "dec_deg",
    "n_valid_bands",
    "n_significant_bands",
    "median_significant_snr",
    "max_abs_opposite_snr",
    "nearest_primary_sep_arcsec",
]


remaining_columns = [
    column
    for column
    in example_candidates.columns
    if column not in first_columns
]


example_candidates = (
    example_candidates[
        first_columns
        + remaining_columns
    ]
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

FINAL_EXAMPLE_FILE = (
    OUT_DIR
    / "DP1_final_example_inspection.csv"
)


example_candidates.to_csv(
    FINAL_EXAMPLE_FILE,
    index=False,
)


display(
    example_candidates[
        first_columns
    ]
)


print("\nSaved:")
print(FINAL_EXAMPLE_FILE)

,candidate_id,paper_candidate_id,catalog_row,sample,ra_deg,dec_deg,n_valid_bands,n_significant_bands,median_significant_snr,max_abs_opposite_snr,nearest_primary_sep_arcsec
0,1,1,15805,ComCam_significant,53.073812,-27.732826,10,2,14.991056,1.279470,1.843890
1,2,2,441278,ComCam_significant,53.593073,-27.926565,8,3,8.452413,1.713349,2.087923
2,3,3,197408,NIR_significant,52.862614,-27.546313,11,5,10.319294,2.643548,7.802467
3,4,4,666836,NIR_significant,52.653276,-27.633525,11,5,16.247038,2.415569,9.236922



Saved:
data/DP1_final_example_inspection.csv


### 11. Measurement and forced photometry for the final examples

Compare the standard measurement (`m`) and forced (`f`) circular-aperture
photometry for the two sources used in the paper figure. In addition to the
individual aperture flags, report the full forced-photometry validity and
significance masks used by the statistical analysis.

In [11]:
# ============================================================
# Compare MEAS and FORCED photometry
# for the two examples used in the paper figure
# ============================================================

CHECK_IDS = [
    1,
    4,
]


selected_examples = (
    example_candidates
    .set_index(
        "candidate_id"
    )
    .loc[
        CHECK_IDS
    ]
    .reset_index()
)


# ------------------------------------------------------------
# Safe scalar readers
# ------------------------------------------------------------

def scalar_float(
    table,
    column,
    row_index,
):

    if column not in table.colnames:
        return np.nan

    value = (
        table[column][row_index]
    )

    if np.ma.is_masked(
        value
    ):
        return np.nan

    try:
        return float(
            value
        )

    except Exception:
        return np.nan


def scalar_flag(
    table,
    column,
    row_index,
):

    if column not in table.colnames:
        return np.nan

    value = (
        table[column][row_index]
    )

    if np.ma.is_masked(
        value
    ):
        return np.nan

    try:
        return bool(
            value
        )

    except Exception:
        return np.nan


# ------------------------------------------------------------
# All eleven bands
# ------------------------------------------------------------

band_groups = (
    [
        (
            "ComCam",
            band,
        )
        for band in COMCAM_BANDS
    ]
    +
    [
        (
            "VIRCAM",
            band,
        )
        for band in VIRCAM_BANDS
    ]
)


# ------------------------------------------------------------
# Build comparison table
# ------------------------------------------------------------

rows = []


for _, candidate in (
    selected_examples.iterrows()
):

    catalog_row = int(
        candidate[
            "catalog_row"
        ]
    )


    for instrument, band in (
        band_groups
    ):

        result = {
            "candidate_id":
                int(
                    candidate[
                        "candidate_id"
                    ]
                ),

            "sample":
                candidate[
                    "sample"
                ],

            "catalog_row":
                catalog_row,

            "band":
                f"{instrument} {band}",
        }


        # ----------------------------------------------------
        # Standard measurement and forced measurement
        # ----------------------------------------------------

        for phot_type, label in [
            (
                "m",
                "meas",
            ),
            (
                "f",
                "forced",
            ),
        ]:

            stem = (
                f"{instrument}_{band}_"
                f"{phot_type}_"
                "base_CircularApertureFlux_6_0"
            )

            flux_col = (
                f"{stem}_flux"
            )

            err_col = (
                f"{stem}_fluxErr"
            )

            flag_col = (
                f"{stem}_flag"
            )


            flux = scalar_float(
                cat,
                flux_col,
                catalog_row,
            )

            flux_err = scalar_float(
                cat,
                err_col,
                catalog_row,
            )

            flag = scalar_flag(
                cat,
                flag_col,
                catalog_row,
            )


            if (
                np.isfinite(
                    flux
                )
                and np.isfinite(
                    flux_err
                )
                and flux_err > 0
            ):

                snr = (
                    flux
                    / flux_err
                )

            else:

                snr = np.nan


            result[
                f"{label}_flux"
            ] = flux

            result[
                f"{label}_fluxErr"
            ] = flux_err

            result[
                f"{label}_snr"
            ] = snr

            result[
                f"{label}_flag"
            ] = flag


        # ----------------------------------------------------
        # Full forced-photometry quality masks
        #
        # These include both the aperture flag and
        # the severe PixelFlags used in the statistics.
        # ----------------------------------------------------

        forced_key = (
            f"{instrument}_{band}"
        )


        result[
            "forced_valid"
        ] = bool(
            forced[
                forced_key
            ]["valid"][
                catalog_row
            ]
        )


        result[
            "forced_significant"
        ] = bool(
            forced[
                forced_key
            ]["significant"][
                catalog_row
            ]
        )


        rows.append(
            result
        )


photometry_comparison = (
    pd.DataFrame(
        rows
    )
)


photometry_comparison = (
    photometry_comparison[
        [
            "candidate_id",
            "sample",
            "catalog_row",
            "band",

            "meas_flux",
            "meas_fluxErr",
            "meas_snr",
            "meas_flag",

            "forced_flux",
            "forced_fluxErr",
            "forced_snr",
            "forced_flag",

            "forced_valid",
            "forced_significant",
        ]
    ]
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

PHOTOMETRY_COMPARISON_FILE = (
    OUT_DIR
    / "DP1_final_example_meas_forced.csv"
)


photometry_comparison.to_csv(
    PHOTOMETRY_COMPARISON_FILE,
    index=False,
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

with pd.option_context(
    "display.max_rows",
    None,

    "display.max_columns",
    None,

    "display.width",
    180,

    "display.float_format",
    lambda x: f"{x:.4g}",
):

    display(
        photometry_comparison
    )


print("\nSaved:")
print(
    PHOTOMETRY_COMPARISON_FILE
)

,candidate_id,sample,catalog_row,band,meas_flux,meas_fluxErr,meas_snr,meas_flag,forced_flux,forced_fluxErr,forced_snr,forced_flag,forced_valid,forced_significant
0,1,ComCam_significant,15805,ComCam u,124.2,88.77,1.399,False,108.7,88.81,1.224,False,True,False
1,1,ComCam_significant,15805,ComCam g,227.6,13.53,16.82,False,227.8,13.52,16.84,False,True,True
2,1,ComCam_significant,15805,ComCam r,241.3,17.91,13.47,False,235.3,17.91,13.14,False,True,True
3,1,ComCam_significant,15805,ComCam i,229.1,32.31,7.089,False,229.1,32.31,7.089,False,False,False
4,1,ComCam_significant,15805,ComCam z,180.4,65.01,2.775,False,199.4,65.09,3.063,False,True,False
5,1,ComCam_significant,15805,ComCam y,350.8,444.4,0.7895,True,364.7,444.6,0.8204,False,True,False
6,1,ComCam_significant,15805,VIRCAM Z,12.15,10.99,1.106,True,12.39,10.98,1.128,False,True,False
7,1,ComCam_significant,15805,VIRCAM Y,1.017,7.189,0.1414,False,-0.3299,7.2,-0.04582,False,True,False
8,1,ComCam_significant,15805,VIRCAM J,7.744,17,0.4556,False,8.967,17.01,0.527,False,True,False
9,1,ComCam_significant,15805,VIRCAM H,26.47,20.23,1.308,True,25.87,20.22,1.279,False,True,False



Saved:
data/DP1_final_example_meas_forced.csv


### 12. Comparison with the Varadaraj et al. (2023) high-redshift sample

We cross-match the six published $z\sim7$ candidates in the ECDF-S
field with the primary sources in the fused DP1 catalogue. For each
matched source, we then test directly whether it satisfies the
optical--NIR extreme-source criteria defined above. This comparison
provides a simple demonstration of whether previously identified
high-redshift candidates are recovered among the populations selected
from the forced-photometry catalogue.

In [12]:
# ============================================================
# Comparison with the Varadaraj et al. (2023) z~7 candidates
# ============================================================

from astropy.coordinates import SkyCoord
import astropy.units as u


# ------------------------------------------------------------
# Published candidates
# ------------------------------------------------------------

varadaraj = pd.DataFrame(
    {
        "id": [
            "VIDEO_z7_23",
            "VIDEO_z7_24",
            "VIDEO_z7_25",
            "VIDEO_z7_26",
            "VIDEO_z7_27",
            "VIDEO_z7_28",
        ],

        "ra_deg": [
            52.572917,
            53.830875,
            53.683667,
            52.122000,
            52.678292,
            53.622208,
        ],

        "dec_deg": [
            -28.239086,
            -27.825597,
            -28.047700,
            -27.991217,
            -27.291775,
            -28.216797,
        ],

        "z_phot": [
            6.52,
            6.57,
            6.59,
            6.67,
            6.67,
            7.38,
        ],
    }
)


# ------------------------------------------------------------
# Coordinates of the primary DP1 catalogue sources
# ------------------------------------------------------------

catalog_ra = np.degrees(
    np.asarray(
        np.ma.filled(
            cat["VIRCAM_K_m_coord_ra"],
            np.nan,
        ),
        dtype=float,
    )
)

catalog_dec = np.degrees(
    np.asarray(
        np.ma.filled(
            cat["VIRCAM_K_m_coord_dec"],
            np.nan,
        ),
        dtype=float,
    )
)


finite_coord = (
    np.isfinite(catalog_ra)
    & np.isfinite(catalog_dec)
)


primary_index = np.flatnonzero(
    primary
    & finite_coord
)


primary_coord = SkyCoord(
    ra=catalog_ra[
        primary_index
    ] * u.deg,

    dec=catalog_dec[
        primary_index
    ] * u.deg,
)


var_coord = SkyCoord(
    ra=varadaraj[
        "ra_deg"
    ].to_numpy() * u.deg,

    dec=varadaraj[
        "dec_deg"
    ].to_numpy() * u.deg,
)


# ------------------------------------------------------------
# Nearest primary DP1 source
# ------------------------------------------------------------

match_index, sep2d, _ = (
    var_coord.match_to_catalog_sky(
        primary_coord
    )
)


matched_rows = (
    primary_index[
        match_index
    ]
)


# Adopt 1 arcsec as the catalogue-match threshold
MATCH_RADIUS_ARCSEC = 1.0

matched = (
    sep2d.arcsec
    < MATCH_RADIUS_ARCSEC
)


# ------------------------------------------------------------
# Build comparison table
# ------------------------------------------------------------

result = varadaraj.copy()


result[
    "catalog_row"
] = matched_rows


result[
    "match_sep_arcsec"
] = sep2d.arcsec


result[
    "matched_within_1arcsec"
] = matched


# ------------------------------------------------------------
# Statistics for the matched DP1 source
#
# Values are reported only for secure (<1 arcsec) matches.
# ------------------------------------------------------------

result[
    "n_ComCam_5sig"
] = np.where(
    matched,
    n_optical_5sig[
        matched_rows
    ],
    np.nan,
)


result[
    "n_VIRCAM_5sig"
] = np.where(
    matched,
    n_nir_5sig[
        matched_rows
    ],
    np.nan,
)


result[
    "valid_griz"
] = np.where(
    matched,
    all_optical_core_valid[
        matched_rows
    ],
    False,
)


result[
    "valid_YJHKs"
] = np.where(
    matched,
    all_nir_core_valid[
        matched_rows
    ],
    False,
)


# ------------------------------------------------------------
# Does the matched source satisfy our extreme selections?
# ------------------------------------------------------------

result[
    "NIR_extreme"
] = np.where(
    matched,
    nir_extreme[
        matched_rows
    ],
    False,
)


result[
    "ComCam_extreme"
] = np.where(
    matched,
    optical_extreme[
        matched_rows
    ],
    False,
)


# ------------------------------------------------------------
# Maximum absolute forced S/N in each wavelength regime
# ------------------------------------------------------------

optical_abs_snr = np.column_stack(
    [
        np.where(
            forced[
                f"ComCam_{band}"
            ]["valid"],
            np.abs(
                forced[
                    f"ComCam_{band}"
                ]["snr"]
            ),
            np.nan,
        )
        for band in COMCAM_BANDS
    ]
)


nir_abs_snr = np.column_stack(
    [
        np.where(
            forced[
                f"VIRCAM_{band}"
            ]["valid"],
            np.abs(
                forced[
                    f"VIRCAM_{band}"
                ]["snr"]
            ),
            np.nan,
        )
        for band in VIRCAM_BANDS
    ]
)


max_abs_ComCam_snr = np.full(
    len(cat),
    np.nan,
)

max_abs_VIRCAM_snr = np.full(
    len(cat),
    np.nan,
)


has_optical_valid = (
    np.isfinite(
        optical_abs_snr
    ).any(axis=1)
)

has_nir_valid = (
    np.isfinite(
        nir_abs_snr
    ).any(axis=1)
)


max_abs_ComCam_snr[
    has_optical_valid
] = np.nanmax(
    optical_abs_snr[
        has_optical_valid
    ],
    axis=1,
)


max_abs_VIRCAM_snr[
    has_nir_valid
] = np.nanmax(
    nir_abs_snr[
        has_nir_valid
    ],
    axis=1,
)


result[
    "max_abs_ComCam_snr"
] = np.where(
    matched,
    max_abs_ComCam_snr[
        matched_rows
    ],
    np.nan,
)


result[
    "max_abs_VIRCAM_snr"
] = np.where(
    matched,
    max_abs_VIRCAM_snr[
        matched_rows
    ],
    np.nan,
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

display_columns = [
    "id",
    "z_phot",
    "ra_deg",
    "dec_deg",
    "catalog_row",
    "match_sep_arcsec",
    "matched_within_1arcsec",
    "n_ComCam_5sig",
    "n_VIRCAM_5sig",
    "max_abs_ComCam_snr",
    "max_abs_VIRCAM_snr",
    "valid_griz",
    "valid_YJHKs",
    "NIR_extreme",
    "ComCam_extreme",
]


with pd.option_context(
    "display.max_columns",
    None,

    "display.width",
    180,

    "display.float_format",
    lambda x: f"{x:.3f}",
):

    display(
        result[
            display_columns
        ]
    )


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print(
    "Varadaraj et al. candidates:"
)

print(
    f"  Published sources             : "
    f"{len(result)}"
)

print(
    f"  DP1 matches within 1 arcsec   : "
    f"{matched.sum()}"
)

print(
    f"  Match NIR-extreme selection   : "
    f"{result['NIR_extreme'].sum()}"
)

print(
    f"  Match ComCam-extreme selection: "
    f"{result['ComCam_extreme'].sum()}"
)

,id,z_phot,ra_deg,dec_deg,catalog_row,match_sep_arcsec,matched_within_1arcsec,n_ComCam_5sig,n_VIRCAM_5sig,max_abs_ComCam_snr,max_abs_VIRCAM_snr,valid_griz,valid_YJHKs,NIR_extreme,ComCam_extreme
0,VIDEO_z7_23,6.520,52.573,-28.239,481087,0.185,True,0.000,4.000,3.152,20.737,True,True,True,False
1,VIDEO_z7_24,6.570,53.831,-27.826,278892,0.155,True,0.000,2.000,NaN,18.413,False,True,False,False
2,VIDEO_z7_25,6.590,53.684,-28.048,616472,0.243,True,0.000,0.000,2.341,NaN,False,False,False,False
3,VIDEO_z7_26,6.670,52.122,-27.991,1367634,197.381,False,NaN,NaN,NaN,NaN,False,False,False,False
4,VIDEO_z7_27,6.670,52.678,-27.292,705400,222.456,False,NaN,NaN,NaN,NaN,False,False,False,False
5,VIDEO_z7_28,7.380,53.622,-28.217,566087,0.053,True,0.000,2.000,1.896,18.182,False,True,False,False


Varadaraj et al. candidates:
  Published sources             : 6
  DP1 matches within 1 arcsec   : 4
  Match NIR-extreme selection   : 1
  Match ComCam-extreme selection: 0


In [16]:
# ============================================================
# Export matched Varadaraj sources for RSP image inspection
# ============================================================

VARADARAJ_RSP_FILE = (
    OUT_DIR
    / "DP1_varadaraj_matched_forced.csv"
)


# Keep only secure DP1 matches.
varadaraj_rsp = (
    result[
        result[
            "matched_within_1arcsec"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


# Preserve the published coordinates separately.
varadaraj_rsp = varadaraj_rsp.rename(
    columns={
        "ra_deg":
            "published_ra_deg",
        "dec_deg":
            "published_dec_deg",
    }
)


# ------------------------------------------------------------
# Use the matched DP1 catalogue position for the image centre
# ------------------------------------------------------------

rows = (
    varadaraj_rsp[
        "catalog_row"
    ]
    .astype(int)
    .to_numpy()
)


varadaraj_rsp[
    "ra_deg"
] = catalog_ra[
    rows
]

varadaraj_rsp[
    "dec_deg"
] = catalog_dec[
    rows
]


# Label used only by the RSP inspection viewer.
varadaraj_rsp[
    "sample"
] = "Varadaraj_z7"


# ------------------------------------------------------------
# Add all eleven forced-photometry S/N values
# ------------------------------------------------------------

for band in COMCAM_BANDS:

    varadaraj_rsp[
        f"snr_ComCam_{band}"
    ] = (
        forced[
            f"ComCam_{band}"
        ]["snr"][
            rows
        ]
    )


for band in VIRCAM_BANDS:

    varadaraj_rsp[
        f"snr_VIRCAM_{band}"
    ] = (
        forced[
            f"VIRCAM_{band}"
        ]["snr"][
            rows
        ]
    )


# ------------------------------------------------------------
# Useful column order
# ------------------------------------------------------------

first_columns = [
    "id",
    "z_phot",
    "catalog_row",
    "match_sep_arcsec",
    "ra_deg",
    "dec_deg",
    "published_ra_deg",
    "published_dec_deg",
    "n_ComCam_5sig",
    "n_VIRCAM_5sig",
    "valid_griz",
    "valid_YJHKs",
    "NIR_extreme",
    "ComCam_extreme",
    "sample",
]


snr_columns = (
    [
        f"snr_ComCam_{band}"
        for band in COMCAM_BANDS
    ]
    +
    [
        f"snr_VIRCAM_{band}"
        for band in VIRCAM_BANDS
    ]
)


varadaraj_rsp = (
    varadaraj_rsp[
        first_columns
        + snr_columns
    ]
)


# ============================================================
# Add formal forced-photometry S/N values
#
# Unlike the statistical snr_* columns, these values are
# flux / fluxErr regardless of the quality flags.
# They are used only for annotation in the image figures.
# ============================================================

rows = (
    varadaraj_rsp[
        "catalog_row"
    ]
    .astype(int)
    .to_numpy()
)


def get_raw_forced_snr(
    instrument,
    band,
    rows,
):

    stem = (
        f"{instrument}_{band}_f_"
        "base_CircularApertureFlux_6_0"
    )

    flux = np.asarray(
        np.ma.filled(
            cat[
                f"{stem}_flux"
            ],
            np.nan,
        ),
        dtype=float,
    )[rows]

    flux_err = np.asarray(
        np.ma.filled(
            cat[
                f"{stem}_fluxErr"
            ],
            np.nan,
        ),
        dtype=float,
    )[rows]


    snr = np.full(
        len(rows),
        np.nan,
    )

    good = (
        np.isfinite(flux)
        & np.isfinite(flux_err)
        & (flux_err > 0)
    )

    snr[good] = (
        flux[good]
        / flux_err[good]
    )

    return snr


for band in COMCAM_BANDS:

    varadaraj_rsp[
        f"raw_snr_ComCam_{band}"
    ] = get_raw_forced_snr(
        "ComCam",
        band,
        rows,
    )

    varadaraj_rsp[
        f"valid_ComCam_{band}"
    ] = (
        forced[
            f"ComCam_{band}"
        ]["valid"][rows]
    )


for band in VIRCAM_BANDS:

    varadaraj_rsp[
        f"raw_snr_VIRCAM_{band}"
    ] = get_raw_forced_snr(
        "VIRCAM",
        band,
        rows,
    )

    varadaraj_rsp[
        f"valid_VIRCAM_{band}"
    ] = (
        forced[
            f"VIRCAM_{band}"
        ]["valid"][rows]
    )

varadaraj_rsp.to_csv(
    VARADARAJ_RSP_FILE,
    index=False,
)


display(
    varadaraj_rsp[
        [
            "id",
            "z_phot",
            "catalog_row",
        ]
        +
        [
            f"raw_snr_ComCam_{band}"
            for band in COMCAM_BANDS
        ]
        +
        [
            f"raw_snr_VIRCAM_{band}"
            for band in VIRCAM_BANDS
        ]
    ]
)

print("\nsaved:")
print(VARADARAJ_RSP_FILE)


,id,z_phot,catalog_row,raw_snr_ComCam_u,raw_snr_ComCam_g,raw_snr_ComCam_r,raw_snr_ComCam_i,raw_snr_ComCam_z,raw_snr_ComCam_y,raw_snr_VIRCAM_Z,raw_snr_VIRCAM_Y,raw_snr_VIRCAM_J,raw_snr_VIRCAM_H,raw_snr_VIRCAM_K
0,VIDEO_z7_23,6.52,481087,-0.341923,-0.100862,1.210064,3.152127,1.207816,-0.070967,NaN,20.736870,11.772830,5.168390,11.085469
1,VIDEO_z7_24,6.57,278892,29.931222,9.876880,12.711491,16.418683,11.820704,21.533054,NaN,18.412920,6.713488,4.139377,0.000030
2,VIDEO_z7_25,6.59,616472,0.559267,0.726198,1.209176,-0.054042,2.340881,1.205399,NaN,NaN,7.459769,12.572106,7.888617
3,VIDEO_z7_28,7.38,566087,-0.241346,0.983666,1.896340,0.353812,0.735647,-0.340556,NaN,18.181624,13.047966,2.786820,4.608823



saved:
data/DP1_varadaraj_matched_forced.csv
